# 021 — SGD com ponderação suave

Parte do SGD do 018: arquitetura 64→32, ReLU, SGD com `momentum=0.9`, taxa selecionada de 0.001, `batch_size=128`, `weight_decay=0` e até 100 épocas. Só varia a ponderação entre 1/frequência (controlo) e 1/√frequência. Escolhe a melhor época de cada versão pela Macro F1 interna e avalia a versão suave nos 20% fixos.

## Protocolo comum

Divisão estratificada fixa 80/20 (`random_state=42`) e validação interna estratificada de 15% dos 80% de treino (`random_state=42`), como nas experiências anteriores. A preparação é ajustada só nas linhas de treino de cada fase e os pesos de classe são calculados só nos respetivos alvos de treino. Todas as escolhas usam Macro F1 interna. A validação fixa já foi consultada em várias experiências; serve apenas como comparação de desenvolvimento, não como avaliação final independente.

Para a submissão, reajustam-se a preparação e modelos novos nas 10.962 linhas rotuladas, recalculando pesos de classe e usando as épocas escolhidas internamente. O CSV copia `RowId` do exemplo oficial na ordem das 3.655 linhas de teste e valida cabeçalho, unicidade, classes inteiras 0–15 e `index=False`.

CSV: `submissions/submission_021_mlp_sgd_ponderacao_suave.csv`. Se ficar abaixo do controlo nos 20% fixos, a submissão é assinalada como exploratória. A pontuação Kaggle fica em branco até ser submetido.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents]
                 if (p / "src" / "experiments_019_021.py").is_file())
sys.path.insert(0, str(repo_root))
from src.experiments_019_021 import run_021, compare_new_submissions, write_new_experiments_report

output = run_021()
result = output["result"]
selection = pd.DataFrame(result["candidates"])[["label", "best_epoch", "internal_macro_f1"]].rename(
    columns={"label": "Ponderação", "best_epoch": "Melhor época", "internal_macro_f1": "Macro F1 interna"}
)
display(selection.style.format({"Macro F1 interna": "{:.4f}"}))
def show_metrics(metrics, title):
    print(title)
    print(f"Macro F1: {metrics['macro_f1']:.4f} | accuracy: {metrics['accuracy']:.4f}")
    per_band = pd.DataFrame(metrics["per_class"]).rename(
        columns={"classe": "Classe", "escalão": "Escalão", "F1": "F1", "recall": "Recall", "support": "Suporte"}
    )
    display(per_band.style.apply(
        lambda row: ["background-color: #fff2cc" if row["Classe"] in (1, 6, 13) else ""] * len(row),
        axis=1,
    ).format({"F1": "{:.4f}", "Recall": "{:.4f}"}))

show_metrics(result["control_metrics"], "Controlo 1/frequência — validação fixa")
show_metrics(result["soft_metrics"], "Variante 1/√frequência — validação fixa")
print(f"Configuração exportada: {result['configuration']}")
print(f"Diferença Macro F1 fixa vs. 018 (0.4365): {result['difference_from_018_fixed_macro_f1']:+.4f}")
print(f"Diferença face ao controlo fixo: {result['difference_from_control_fixed_macro_f1']:+.4f}")
print(f"Classificação exploratória: {result['exploratory']}")
print(f"CSV: {result['submission']}")

comparisons = compare_new_submissions(repo_root)
display(comparisons)
report_path = write_new_experiments_report(repo_root)
print(f"Relatório comparativo: {report_path.relative_to(repo_root)}")


,Ponderação,Melhor época,Macro F1 interna
0,1/frequência,93,0.4784
1,1/sqrt(frequência),100,0.4775


Controlo 1/frequência — validação fixa
Macro F1: 0.4365 | accuracy: 0.4364


,Classe,Escalão,F1,Recall,Suporte
0,0,15–17 anos,0.6667,0.5854,41
1,1,18–19 anos,0.5143,0.7941,34
2,2,20–24 anos,0.4132,0.3125,80
3,3,25–29 anos,0.4123,0.6267,75
4,4,30–34 anos,0.1757,0.1444,90
5,5,35–39 anos,0.3665,0.3566,129
6,6,40–44 anos,0.3178,0.2965,172
7,7,45–49 anos,0.3700,0.3988,173
8,8,50–54 anos,0.5801,0.5304,181
9,9,55–59 anos,0.6025,0.5862,203


Variante 1/√frequência — validação fixa
Macro F1: 0.4250 | accuracy: 0.4350


,Classe,Escalão,F1,Recall,Suporte
0,0,15–17 anos,0.7089,0.6829,41
1,1,18–19 anos,0.5102,0.7353,34
2,2,20–24 anos,0.3770,0.2875,80
3,3,25–29 anos,0.3605,0.4133,75
4,4,30–34 anos,0.3131,0.3444,90
5,5,35–39 anos,0.3841,0.4109,129
6,6,40–44 anos,0.1709,0.1163,172
7,7,45–49 anos,0.3598,0.3931,173
8,8,50–54 anos,0.6310,0.6851,181
9,9,55–59 anos,0.6031,0.5764,203


Configuração exportada: MLP 64→32, ReLU, SGD lr=0.001, momentum=0.9, pesos 1/sqrt(frequência), época=100
Diferença Macro F1 fixa vs. 018 (0.4365): -0.0115
Diferença face ao controlo fixo: -0.0115
Classificação exploratória: True
CSV: submissions/submission_021_mlp_sgd_ponderacao_suave.csv


,csv_a,csv_b,previsoes_diferentes,duplicado
0,submission_019_ensemble_pesos.csv,submission_001_regressao_logistica.csv,1263,False
1,submission_019_ensemble_pesos.csv,submission_002_regressao_logistica_regularizad...,1043,False
2,submission_019_ensemble_pesos.csv,submission_004_arvore_decisao.csv,2123,False
3,submission_019_ensemble_pesos.csv,submission_007_xgboost.csv,734,False
4,submission_019_ensemble_pesos.csv,submission_010_mlp_pytorch.csv,1136,False
5,submission_019_ensemble_pesos.csv,submission_012_mlp_perda_ponderada.csv,1142,False
6,submission_019_ensemble_pesos.csv,submission_014_xgboost_mlp.csv,433,False
7,submission_019_ensemble_pesos.csv,submission_015_mlp_learning_rate.csv,829,False
8,submission_019_ensemble_pesos.csv,submission_016_xgboost_mlp015.csv,291,False
9,submission_019_ensemble_pesos.csv,submission_017_mlp_ponderacao_suave.csv,858,False


Relatório comparativo: reports/experiments_019_021.md


## Conclusão e Decisão
- Data: 12/11
- Pontuação Kaggle: 0.43
- Conclusão: A ponderação suave com SGD ficou abaixo do 018 tanto localmente (0,4250 vs. 0,4365) como no Kaggle (0,43 vs. 0,44). Esta combinação não melhorou os resultados.
- Melhorias: Voltar à ponderação inversa do 018 e afinar a taxa do SGD perto de 0,001: 0,0005, 0,001 e 0,002.